# fetchr — Data Audit Notebook (109-field model)

## Purpose

Before doing any machine learning, we need to understand what our data actually looks like. This notebook answers:
- How many dogs do we have?
- Which fields are missing data (nulls), and how often?
- What values do categorical fields like `size` and `breed` contain?
- How reliable are the boolean fields like `good_with_kids`?
- Which new fields (`vaccinated`, `activity_level`, `requires_fenced_yard`) are actually populated?
- How useful is the free-text `description` and the `personality_traits` list?

The answers tell us which fields we can trust as matching signals and which ones need special handling.

**Dataset:** 100 dogs from PetFinder (NJ/Jersey City area). Model expanded from 31 → 109 fields in May 2026.

---

## What this notebook covers

Of the 109 fields in the model, **26 are 100% null** (PetFinder never sends them) — leaving **83 fields that have at least some data**. But not all 83 matter for matching. This notebook focuses on the subset that does:

| Category | Fields audited |
|---|---|
| Categorical | `size`, `age_category`, `gender`, `status`, `coat_length`, `color` |
| Behavior booleans | `good_with_kids`, `good_with_dogs`, `good_with_cats`, `house_trained`, `vaccinated`, `spayed_neutered`, `special_needs` |
| Age derivation | `age_label`, `age_range_label`, `birth_date` → used to populate `age_years_approx` |
| List fields | `personality_traits` |
| Text | `description` |
| Numeric | `weight_min`, `weight_max` |

**Fields intentionally excluded from this audit** — fully populated but not used for matching:
- Identifiers: `id`, `source_id`, `org_id`, `contact_id`, `location_id`, `org_display_id`
- URLs: `petfinder_url`, `source_url`, `org_adoption_url`, `org_website`
- Org metadata: `org_type`, `shelter_name`, `org_social_urls`, `org_special_services`
- Display fields: `photos`, `city`, `state`, `zip`, `adoption_fee`

The full classification of all 83 fields (by type, null rate, and matching signal verdict) is in the **Field Inventory** cell immediately below. The §9 Summary cell is the output of the full analysis — it tells you which fields are ready to use, which need special handling, and which to exclude from the matching algorithm.

## Field Inventory — All 83 Populated Fields

Of the 109 fields in the model, 26 are 100% null (PetFinder never sends them). The remaining 83 have at least some data and are categorised below with their null rate and whether they are useful for the matching algorithm.

---

### Boolean (17 fields)

| Field | Null% | Matching signal? |
|---|---|---|
| `is_mixed` | 0% | Yes — soft filter |
| `special_needs` | 0% | Yes — hard filter |
| `is_appt_only` | 0% | No — shelter logistics |
| `is_map_hidden` | 0% | No — shelter logistics |
| `is_public_location` | 0% | No — shelter logistics |
| `vaccinated` | 3% | Yes — soft filter |
| `spayed_neutered` | 7% | Yes — soft filter |
| `house_trained` | 16% | Yes — hard filter |
| `good_with_dogs` | 28% | Yes — hard filter |
| `good_with_kids` | 31% | Yes — hard filter |
| `display_adoption_fee` | 53% | No — display flag |
| `adoption_fee_waived` | 55% | No — financial |
| `private_address` | 58% | No — shelter privacy |
| `good_with_cats` | 73% | Yes — soft filter (sparse) |
| `org_onsite_vet` | 73% | Maybe |
| `org_supports_rehome` | 73% | No |
| `good_with_other_animals` | 93% | No — too sparse |

---

### Categorical (33 fields)

| Field | Null% | Matching signal? |
|---|---|---|
| `size` | 0% | Yes |
| `age_category` | 0% | Yes |
| `age_label` | 0% | Yes (same as age_category) |
| `age_range_label` | 0% | Yes (for age_years_approx derivation) |
| `gender` | 0% | Yes — soft filter |
| `status` | 0% | Yes — hard filter |
| `weight_range_label` | 0% | Redundant with size |
| `color` | 6% | Maybe — soft filter |
| `coat_length` | 28% | Yes — soft filter |
| `color_secondary` | 49% | No |
| `breed_secondary` | 84% | No — too sparse |
| `org_spay_neuter_policy` | 96% | No |
| `color_tertiary` | 97% | No — too sparse |
| `source` | 0% | No — always "petfinder" |
| `animal_type` | 0% | No — always "Dog" |
| `species` | 0% | No — always "Dog" |
| `country` | 0% | No — always "US" |
| `org_type` | 0% | No |
| `state` | 0% | Display only |
| `city` | 0% | Display only |
| `zip` | 0% | Display only (+ geocoding input) |
| `location_type` | 0% | No |
| `location_id`, `location_name`, `location_email` | 0% | No — identifiers |
| `location_phone` | 40% | No |
| `location_street` | 49% | No |
| `shelter_name` | 0% | Display only |
| `org_id`, `org_website`, `org_display_id` | 0% | Display only |
| `org_adoption_url` | 4% | Display only |
| `org_custom_url_alias` | 69% | No |
| `contact_id` | 0% | No — identifier |
| `org_mission_statement` | 1% | No |

---

### List (6 fields)

| Field | Null% | Matching signal? |
|---|---|---|
| `personality_traits` | 0% | Yes — multi-hot + embed |
| `photos` | 0% | Display only |
| `media_records` | 0% | Display only |
| `org_social_urls` | 0% | No |
| `org_special_services` | 0% | No |
| `tags` | 0% | No — key present, always null |

---

### String / Free Text (8 fields)

| Field | Null% | Matching signal? |
|---|---|---|
| `description` | 0% | Yes — sentence embedding |
| `breed_primary` | 0% | Yes — soft filter |
| `name` | 0% | Display only |
| `petfinder_url`, `source_url`, `sponsor_a_pet_url` | 0% | Display only |
| `id`, `source_id` | 0% | Identifiers |
| `petfinder_notes` | 79% | No |

---

### Numeric (9 fields)

| Field | Null% | Matching signal? |
|---|---|---|
| `weight_min` | 0% | Yes — range filter |
| `weight_max` | 0% | Yes — range filter |
| `adoption_fee` | 2% | Display only |
| `org_adoption_fee_min`, `org_adoption_fee_max` | 73% | No |
| `org_annual_adoptions`, `org_annual_intake` | 73% | No |
| `org_foster_count`, `org_employee_count`, `org_volunteer_count` | 73% | No |

---

### Datetime (6 fields)

| Field | Null% | Matching signal? |
|---|---|---|
| `first_seen_at`, `last_updated_at` | 0% | Ops / freshness tracking |
| `birth_date` | 71% | Yes — derive age_years_approx |
| `adoption_date` | 98% | History tracking |
| `adoption_status_change_date` | 79% | History tracking |
| `intake_date` | 53% | No |

## 1. Load the data

`pandas` is a library that loads tabular data into a structure called a **DataFrame** — essentially a programmable spreadsheet where each row is a dog and each column is a field.

`json_normalize` handles the fact that our data is a list of JSON objects — it flattens each object into a row.

In [1]:
import json
import pandas as pd

with open('fetchr.json') as f:
    raw = json.load(f)

df = pd.json_normalize(raw)

print(f'Rows (dogs): {len(df)}')
print(f'Columns (fields): {len(df.columns)}')
print(f'\nAll fields:\n{list(df.columns)}')

Rows (dogs): 100
Columns (fields): 109

All fields:
['id', 'source', 'source_id', 'source_url', 'name', 'animal_type', 'microchip_id', 'internal_notes', 'match_label', 'out_of_town', 'import_updates_enabled', 'import_deletes_enabled', 'breed_primary', 'breed_secondary', 'is_mixed', 'age_category', 'age_years_approx', 'age_label', 'age_range_label', 'size', 'weight_min', 'weight_max', 'weight_range_label', 'gender', 'color', 'color_secondary', 'color_tertiary', 'coat_length', 'declawed', 'species', 'spayed_neutered', 'vaccinated', 'special_needs', 'special_needs_notes', 'birth_date', 'house_trained', 'activity_level', 'requires_fenced_yard', 'knows_basic_commands', 'behavior_other_animals', 'good_with_kids', 'good_with_dogs', 'good_with_cats', 'good_with_other_animals', 'personality_traits', 'location_id', 'location_name', 'location_type', 'location_contact_name', 'location_email', 'location_phone', 'is_appt_only', 'is_map_hidden', 'is_public_location', 'private_address', 'location_stre

## 2. Null audit — which fields are missing data?

A **null** means the scraper found no value for that field on PetFinder. 
This matters enormously for ML: if a field is null 80% of the time, it's not a reliable feature.

The table below shows, for each field:
- **null_count** — how many dogs are missing this value
- **null_pct** — what percentage of all dogs that is
- **filled_pct** — the inverse — how much of the field is actually usable

In [2]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

null_audit = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct,
    'filled_pct': (100 - null_pct),
}).sort_values('null_pct', ascending=False)

pd.set_option('display.max_rows', None)
filtered = null_audit[null_audit['null_count'] > 0]
print(f'Total fields: {len(null_audit)}')
print(f'Fields with at least one null: {len(filtered)}')
filtered

Total fields: 109
Fields with at least one null: 62


,null_count,null_pct,filled_pct
activity_level,100,100.0,0.0
lat,100,100.0,0.0
requires_fenced_yard,100,100.0,0.0
knows_basic_commands,100,100.0,0.0
behavior_other_animals,100,100.0,0.0
extended_description,100,100.0,0.0
location_contact_name,100,100.0,0.0
contact_phone,100,100.0,0.0
contact_last_name,100,100.0,0.0
contact_first_name,100,100.0,0.0


## 3. Categorical field distributions

For fields like `size`, `age_category`, `breed`, and `gender`, we want to know:
- What values actually appear in the data?
- Are they evenly spread or heavily skewed toward one value?

Just a list of column names you want to analyze. Defining it upfront means you can add/remove fields in one place instead of hunting through the code.

A heavily skewed field (e.g., 95% of dogs are "medium" size) is a weak ML feature — it doesn't help the model distinguish between dogs.

#### Understanding the Code:
```df[field].value_counts(dropna=False)
df[field] — selects one column (a Series)
.value_counts() — counts how many times each unique value appears, sorted descending automatically
dropna=False — includes nulls in the count rather than silently ignoring them
```

<mark>This is the critical flag. Without it, if 10 dogs have no size recorded, those rows disappear from your count and your percentages look cleaner than reality. dropna=False forces honesty.</mark>

In [3]:
categorical_fields = ['size', 'age_category', 'gender', 'status', 'coat_length', 'activity_level', 'color', 'weight_range_label']

for field in categorical_fields:
    print(f'\n--- {field} ---')
    counts = df[field].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(1)
    print(pd.DataFrame({'count': counts, 'pct': pct}).to_string())


--- size ---
        count   pct
size               
medium     74  74.0
large      15  15.0
small      11  11.0

--- age_category ---
              count   pct
age_category             
adult            48  48.0
young            37  37.0
puppy             9   9.0
senior            6   6.0

--- gender ---
        count   pct
gender             
female     54  54.0
male       46  46.0

--- status ---
           count   pct
status                
available     98  98.0
adopted        2   2.0

--- coat_length ---
             count   pct
coat_length             
Short           65  65.0
None            28  28.0
Medium           6   6.0
Curly            1   1.0

--- activity_level ---
                count    pct
activity_level              
None              100  100.0

--- color ---
                                  count   pct
color                                        
Black                                23  23.0
White / Cream                        16  16.0
Brown / Chocolate      

## 3a. Age fields — raw values and the age_years_approx gap

`age_label` and `age_range_label` are the raw strings PetFinder sends in `physical.age` before any normalization. Understanding what values actually arrive is the prerequisite for implementing `age_years_approx`.

`age_years_approx` is currently **100% null** — not because PetFinder doesn't send age data, but because we haven't implemented the derivation step yet. The path is:
1. Parse `age_range_label` (e.g. `"1 - 2 Years"`) to extract a numeric midpoint → `age_years_approx`
2. For dogs that have `birth_date`, calculate exact age from DOB instead — more precise than the range label

In [4]:
print('--- age_label ---')
print(df['age_label'].value_counts(dropna=False).to_string())

print('\n--- age_range_label ---')
print(df['age_range_label'].value_counts(dropna=False).to_string())

print('\n--- age_years_approx ---')
print(f'Populated: {df["age_years_approx"].notna().sum()} / {len(df)}')
print('100% null — needs derivation from age_range_label (or birth_date where available)')

--- age_label ---
age_label
Adult     48
Young     37
Puppy      9
Senior     6

--- age_range_label ---
age_range_label
(3-8 years)           48
(1-3 years)           37
(less than 1 year)     9
(8+ years)             6

--- age_years_approx ---
Populated: 0 / 100
100% null — needs derivation from age_range_label (or birth_date where available)


## Reading the Output

**What this tells you for ML (100-dog dataset)**

| Field | Verdict |
|-------|---------|
| `status` | **Critical hard filter** — 98% available, 2% adopted. The matching engine must always filter `WHERE status = 'available'` before any other logic. |
| `size` | **Decent** — PetFinder defines 4 values (Small/Medium/Large/Extra Large); only 3 appear in this 100-dog sample (no Extra Large). Keep all 4 in the encoding plan — Extra Large will appear with a larger dataset and the encoder must handle it. |
| `age_category` | **Good** — spread across 4 values |
| `gender` | **Good** — near 50/50 |
| `color` | **Usable as soft filter** — 94% populated, 11 distinct values. Some adopters have color preferences; treat as a low-weight soft filter, not a hard filter. |
| `coat_length` | **Usable** — ~72% populated; null = unknown, not a value |
| `activity_level` | **UNUSABLE** — 0% populated by PetFinder. Never set. Do not use as a matching filter. |

`activity_level` being 0% is the most important finding from the categorical section. It was in our Tier 2 matching fields plan — but PetFinder never sends it. The field is stored for future use if PetFinder adds it, but it cannot be used for matching now.

## 4. Breed distribution

Breed is special — it likely has many unique values (high cardinality). 
<mark>High cardinality makes one-hot encoding impractical (you'd end up with hundreds of columns).</mark> One-hot encoding converts each unique value into its own column, filled with 0s and 1s.
This section shows the top breeds and how many unique breeds we have total.

In [5]:
print(f'Unique primary breeds: {df["breed_primary"].nunique()}')
print(f'\nTop 15 breeds:')
print(df['breed_primary'].value_counts().head(15).to_string())

Unique primary breeds: 24

Top 15 breeds:
breed_primary
Mixed Breed                       30
Pit Bull Terrier                  28
American Staffordshire Terrier     8
Shepherd                           5
Labrador Retriever                 3
Hound                              3
Black Labrador Retriever           2
Yorkshire Terrier                  2
Boston Terrier                     2
American Bulldog                   2
German Shepherd Dog                2
Dogo Argentino                     1
Shih Tzu                           1
Catahoula Leopard Dog              1
Siberian Husky                     1


## 5. Boolean field distributions — true / false / unknown

The boolean fields (`good_with_kids`, `good_with_dogs`, `good_with_cats`, `house_trained`) are critical for matching — a user searching for a dog good with kids cares a lot about this.

<mark>But remember: **null ≠ false**. A null means PetFinder didn't specify. We need to see how many dogs have a known answer vs. unknown.</mark>

In [6]:
boolean_fields = [
    'good_with_kids', 'good_with_dogs', 'good_with_cats', 'good_with_other_animals',
    'house_trained', 'requires_fenced_yard',
    'vaccinated', 'spayed_neutered',
    'special_needs', 'is_mixed',
]

for field in boolean_fields:
    counts = df[field].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(1)
    summary = pd.DataFrame({'count': counts, 'pct': pct})
    summary.index = summary.index.map(lambda x: 'unknown/null' if pd.isna(x) else ('yes' if x else 'no'))
    print(f'\n--- {field} ---')
    print(summary.to_string())


--- good_with_kids ---
                count   pct
good_with_kids             
yes                62  62.0
unknown/null       31  31.0
no                  7   7.0

--- good_with_dogs ---
                count   pct
good_with_dogs             
yes                58  58.0
unknown/null       28  28.0
no                 14  14.0

--- good_with_cats ---
                count   pct
good_with_cats             
unknown/null       73  73.0
yes                14  14.0
no                 13  13.0

--- good_with_other_animals ---
                         count   pct
good_with_other_animals             
unknown/null                93  93.0
no                           7   7.0

--- house_trained ---
               count   pct
house_trained             
yes               66  66.0
no                18  18.0
unknown/null      16  16.0

--- requires_fenced_yard ---
                      count    pct
requires_fenced_yard              
unknown/null            100  100.0

--- vaccinated ---
              

## Boolean Field Findings — Key Signals vs. Gaps

| Field | Coverage | Verdict for matching |
|---|---|---|
| `vaccinated` | ~97% | **Ready** — use as a soft filter or display signal |
| `spayed_neutered` | ~93% | **Ready** — use as a soft or hard filter |
| `good_with_dogs` | ~72% | **Usable** — treat null as "unknown", not "no" |
| `house_trained` | ~82% | **Usable** — most dogs have a known value |
| `good_with_kids` | ~69% | **Usable with caution** — 31% null |
| `good_with_cats` | ~27% | **Risky** — 73% null; don't use as a hard filter |
| `good_with_other_animals` | ~7% | **UNUSABLE** — almost never set |
| `requires_fenced_yard` | ~0% | **UNUSABLE** — PetFinder never populates this field |
| `special_needs` | ~100% | **Ready** — always set (default false when not specified) |
| `is_mixed` | ~100% | **Ready** |

**Critical:** `requires_fenced_yard` is 0% populated — same as `activity_level`. Both were in the matching plan as Tier 1/2 signals, but PetFinder doesn't send them. Remove them from any hard filter implementation.

**On null ≠ false:** For all the boolean behavior fields, a null means the shelter didn't specify — NOT that the answer is false. When building the matching engine, a family asking for a dog that's good with kids should only see dogs where `good_with_kids IS TRUE`, not dogs where `good_with_kids IS NULL`.

## 6. Personality traits — what does the shelter say about each dog?

`personality_traits` comes from `behavior.personalityTraits` in PetFinder's JSON — a curated tag list shelters apply when creating a listing. It's the most actionable semantic signal we have *before* embedding descriptions.

This section unpacks all trait lists, counts every unique trait, and tells us:
- Coverage: what % of dogs have at least one trait?
- Vocabulary: is the tag set small and normalized (good for ML), or freeform chaos (needs cleaning)?

**Why personality_traits and not tags?** The model also has a `tags` field from a different part of the PetFinder JSON. `personality_traits` is specifically the shelter's behavioral assessment — the curated vocabulary. We analyze both below.

In [7]:
from collections import Counter

def audit_list_field(df, field_name):
    all_values = [v for lst in df[field_name] if isinstance(lst, list) for v in lst]
    counts = Counter(all_values)
    dogs_with_any = df[field_name].apply(lambda t: isinstance(t, list) and len(t) > 0).sum()
    print(f'Dogs with at least one {field_name}: {dogs_with_any} / {len(df)} ({dogs_with_any/len(df)*100:.1f}%)')
    print(f'Total unique values: {len(counts)}')
    print(f'\nTop 25 by frequency:')
    for val, count in counts.most_common(25):
        print(f'  {count:3d}x  {val}')

print('=' * 60)
print('PERSONALITY TRAITS (behavior.personalityTraits)')
print('=' * 60)
audit_list_field(df, 'personality_traits')

empty_list_count = df['personality_traits'].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()
print(f'\nDogs with empty personality_traits list: {empty_list_count} ({empty_list_count/len(df)*100:.1f}%)')
print('Note: isnull() misses these — they are [] not None, but equally unusable.')

print()
print('=' * 60)
print('TAGS (separate PetFinder field)')
print('=' * 60)
audit_list_field(df, 'tags')
print('Note: tags is 0% populated — PetFinder does not send this field. Do not use for matching.')

PERSONALITY TRAITS (behavior.personalityTraits)
Dogs with at least one personality_traits: 85 / 100 (85.0%)
Total unique values: 51

Top 25 by frequency:
   57x  Affectionate
   56x  Friendly
   47x  Playful
   43x  Curious
   39x  Funny
   34x  Athletic
   28x  Smart
   23x  Gentle
   17x  Quiet
   15x  Housetrained
   15x  Brave
   14x  Loyal
   14x  Good with Dogs
   13x  Loves Kisses
   12x  Crate Trained
   10x  Good with Kids
   10x  Dignified
   10x  Independent
    8x  Couch Potato
    7x  Good with Cats
    6x  Couch
    6x  Loves
    4x  Loves Food
    3x  Protective
    3x  Exercise Needs   Weekend Adventurer

Dogs with empty personality_traits list: 15 (15.0%)
Note: isnull() misses these — they are [] not None, but equally unusable.

TAGS (separate PetFinder field)
Dogs with at least one tags: 0 / 100 (0.0%)
Total unique values: 0

Top 25 by frequency:
Note: tags is 0% populated — PetFinder does not send this field. Do not use for matching.


## 7. Description audit — how useful is the free text?

`description` is the richest potential signal for semantic matching — it's the paragraph a shelter writes about the dog's personality. But it's only useful if:
- Enough dogs have one (coverage)
- They're long enough to contain real signal (length)

A 3-word description like "Sweet, gentle dog" is not very useful for embeddings. A 5-sentence paragraph is.

**Interview angle**

This is a data quality audit pattern — a standard first step before using any text field. If asked in an interview, frame it as: "Before treating a column as usable, I validate both nullability and semantic emptiness, then characterize the distribution with median rather than mean to account for skew." A follow-up might be: "How would you decide if description quality is good enough to use for search/ranking?" — answer: look at coverage % and median length; if >80% have descriptions and median >50 chars, it's probably usable.

In [8]:
has_description = df['description'].notna() & (df['description'].str.strip() != '')
desc_lengths = df.loc[has_description, 'description'].str.len()

print(f'Dogs with a description: {has_description.sum()} / {len(df)} ({has_description.mean()*100:.1f}%)')

if len(desc_lengths) > 0:
    print(f'\nDescription length (characters):')
    print(f'  shortest : {desc_lengths.min()}')
    print(f'  median   : {desc_lengths.median():.0f}')
    print(f'  longest  : {desc_lengths.max()}')
    print(f'\nSample description (first dog that has one):')
    sample = df.loc[has_description, 'description'].iloc[0]
    print(f'  "{sample[:300]}..."' if len(sample) > 300 else f'  "{sample}"')

Dogs with a description: 100 / 100 (100.0%)

Description length (characters):
  shortest : 320
  median   : 980
  longest  : 2911

Sample description (first dog that has one):
  "VIOLET IS BEING FOSTERED IN HOUSTON, TX.  OUT OF STATE TRANSPORTATION CAN BE ARRANGED
********************************************************************************************************

MEET VIOLET!

VIOLET was on the kill-list at a Houston shelter because she had demodex, which is a non conta..."


In [9]:
print(f'weight_min coverage: {df["weight_min"].notna().sum()} / {len(df)} ({df["weight_min"].notna().mean()*100:.1f}%)')
print(f'weight_max coverage: {df["weight_max"].notna().sum()} / {len(df)} ({df["weight_max"].notna().mean()*100:.1f}%)')

has_weight = df['weight_min'].notna()
print(f'\nweight_min (lbs):')
print(f'  min      : {df.loc[has_weight, "weight_min"].min()}')
print(f'  median   : {df.loc[has_weight, "weight_min"].median():.0f}')
print(f'  mean     : {df.loc[has_weight, "weight_min"].mean():.1f}')
print(f'  max      : {df.loc[has_weight, "weight_min"].max()}')
print('  Note: min=0 is not a data error — PetFinder uses 0 as the lower bound of the Small band (0-25 lbs).')

print(f'\nweight_max (lbs):')
print(f'  min      : {df.loc[has_weight, "weight_max"].min()}')
print(f'  median   : {df.loc[has_weight, "weight_max"].median():.0f}')
print(f'  mean     : {df.loc[has_weight, "weight_max"].mean():.1f}')
print(f'  max      : {df.loc[has_weight, "weight_max"].max()}')

print(f'\nWeight range cross-tab with size label:')
print('Note: min=median=max per size group is expected — PetFinder sends fixed weight bands,')
print('not measured weights. Every dog of the same size gets the same range.')
df['weight_mid'] = (df['weight_min'] + df['weight_max']) / 2
print(df.groupby('size')['weight_mid'].describe()[['min','50%','max']].rename(columns={'50%':'median'}).to_string())

weight_min coverage: 100 / 100 (100.0%)
weight_max coverage: 100 / 100 (100.0%)

weight_min (lbs):
  min      : 0
  median   : 26
  mean     : 28.4
  max      : 61
  Note: min=0 is not a data error — PetFinder uses 0 as the lower bound of the Small band (0-25 lbs).

weight_max (lbs):
  min      : 25
  median   : 60
  mean     : 62.1
  max      : 100

Weight range cross-tab with size label:
Note: min=median=max per size group is expected — PetFinder sends fixed weight bands,
not measured weights. Every dog of the same size gets the same range.
         min  median   max
size                      
large   80.5    80.5  80.5
medium  43.0    43.0  43.0
small   12.5    12.5  12.5


## 8. Birth date — null audit and age validation

`birth_date` has ~29% coverage. For the dogs where it's populated, we can calculate their exact age and cross-check it against PetFinder's `age_category` and `age_range_label`.

If the calculated age consistently falls within the expected band, it confirms `age_category` is reliable for the full dataset — not just the 29% with a DOB. Mismatches flag shelters that entered stale or incorrect age data.

PetFinder's age band boundaries:
- **Puppy**: < 1 year
- **Young**: 1–3 years
- **Adult**: 3–8 years
- **Senior**: > 8 years

In [10]:
from datetime import date

print(f'birth_date populated: {df["birth_date"].notna().sum()} / {len(df)} ({df["birth_date"].notna().mean()*100:.1f}%)')

has_dob = df['birth_date'].notna()

if has_dob.sum() > 0:
    today = pd.Timestamp(date.today())
    df.loc[has_dob, 'age_from_dob'] = (
        (today - pd.to_datetime(df.loc[has_dob, 'birth_date'])).dt.days / 365.25
    ).round(1)

    def dob_to_expected_category(age_years: float) -> str:
        if age_years < 1:
            return 'puppy'
        elif age_years < 3:
            return 'young'
        elif age_years <= 8:
            return 'adult'
        else:
            return 'senior'

    df.loc[has_dob, 'expected_category'] = df.loc[has_dob, 'age_from_dob'].apply(dob_to_expected_category)
    mismatches = df.loc[has_dob & (df['age_category'] != df['expected_category'])]

    print(f'\nMismatches: {len(mismatches)} / {has_dob.sum()} dogs with a birth_date')
    if len(mismatches) > 0:
        print(mismatches[['name', 'age_from_dob', 'expected_category', 'age_category']].to_string())
    else:
        print('All DOB-derived ages agree with age_category — field is trustworthy.')

    print(f'\nFull DOB validation table:')
    print(df.loc[has_dob, ['name', 'birth_date', 'age_from_dob', 'age_category', 'age_range_label']].to_string())

birth_date populated: 29 / 100 (29.0%)

Mismatches: 11 / 29 dogs with a birth_date
            name  age_from_dob expected_category age_category
2          Russo           2.5             young        adult
17          Ruby           2.5             young        adult
75          Sage           1.6             young        adult
77        Goggle           0.6             puppy        young
81        Cooler           0.7             puppy        young
84         Wilma           2.1             young        adult
85        PB & J           1.6             young        adult
87       Chowder           0.8             puppy        young
89          Fran           2.3             young        adult
90         Corto           1.9             young        adult
94  Lobster Roll           0.8             puppy        young

Full DOB validation table:
                name           birth_date  age_from_dob age_category     age_range_label
2              Russo  2023-11-20T00:00:00           2.5 

## 9. Summary — What We Learned + Updated Matching Plan

### Fields confirmed as reliable matching signals (109-field dataset, 100 dogs)

| Field | Coverage | Encoding plan |
|---|---|---|
| `status` | 100% | Hard SQL filter — always `WHERE status = 'available'`; never show pending/adopted dogs |
| `size` | 100% | One-hot (Small/Medium/Large/Extra Large) — only 3 in current sample, keep all 4 for larger scrapes |
| `age_category` | 100% | One-hot (puppy/young/adult/senior) — treat as soft filter; 38% mismatch rate vs DOB-derived age (see §8) |
| `gender` | 100% | One-hot (male/female) |
| `breed_primary` | 100% | Group into ~10 breed families; one-hot |
| `weight_min` / `weight_max` | 100% | Use directly as numeric range filter |
| `is_mixed` | 100% | Binary |
| `special_needs` | 100% | Binary |
| `spayed_neutered` | ~93% | Binary; treat null as "unknown" |
| `vaccinated` | ~97% | Binary; treat null as "unknown" |
| `house_trained` | ~82% | Binary + known flag |
| `good_with_dogs` | ~72% | Binary + known flag |
| `coat_length` | ~72% | Categorical; treat null as "unknown" |
| `good_with_kids` | ~69% | Binary + known flag |
| `color` | ~94% | Categorical; treat null as "unknown"; soft filter only |
| `good_with_cats` | ~27% | **Risky** — use as a soft filter only |
| `personality_traits` | ~85% | Multi-hot; also embed as text signal |
| `description` | 100% | Sentence embedding (all-MiniLM-L6-v2, 384-dim) |

**On `age_years_approx`:** Currently 100% null — not yet derived. When implemented, the derivation priority is: (1) calculate from `birth_date` where available (29% of dogs, more precise), (2) parse midpoint from `age_range_label` for the rest. Until then, use `age_category` as the age signal, treating it as a soft filter given the 38% mismatch rate found in §8.

### Fields removed from the matching plan (not populated by PetFinder)

| Field | Coverage | Status |
|---|---|---|
| `activity_level` | 0% | Stored in DB, never used for matching |
| `requires_fenced_yard` | 0% | Stored in DB, never used for matching |
| `good_with_other_animals` | ~7% | Too sparse to rely on |
| `tags` | 0% | PetFinder does not send this field |

These were in the original Tier 1/2 plan but PetFinder doesn't populate them. They're kept in the schema for future use if PetFinder adds them, but the matching algorithm should not use them as hard filters.

### Next: Step 3 — Migrate to Postgres
SQLite is unblocking for data collection, but Postgres + pgvector is required before any ML work.
See `notes/expansion-plan.md`.

In [11]:
feature_fields = [
    'status',
    'size', 'age_category', 'gender', 'breed_primary', 'coat_length', 'color',
    'weight_min', 'weight_max', 'is_mixed', 'spayed_neutered', 'vaccinated',
    'good_with_kids', 'good_with_dogs', 'good_with_cats',
    'house_trained', 'special_needs',
    'personality_traits', 'description',
]

summary_rows = []
for field in feature_fields:
    null_pct = df[field].isnull().sum() / len(df) * 100
    is_list = df[field].apply(lambda x: isinstance(x, list)).any()
    unique = '—' if is_list else df[field].nunique()
    summary_rows.append({'field': field, 'null_%': round(null_pct, 1), 'unique_values': unique})

pd.DataFrame(summary_rows).set_index('field')

,null_%,unique_values
field,,
status,0.0,2
size,0.0,3
age_category,0.0,4
gender,0.0,2
breed_primary,0.0,24
coat_length,28.0,3
color,6.0,11
weight_min,0.0,3
weight_max,0.0,3


## 10. Feature Engineering Recommendations

Based on what we found above, here's the encoding plan for each matching-signal field:

| Field | Type | Plan |
|---|---|---|
| `status` | Categorical | Hard SQL filter — `WHERE status = 'available'`; not encoded as an ML feature |
| `size` | Categorical | One-hot encode (Small/Medium/Large/Extra Large) — all 4 values must be in the encoder even though only 3 appear in this sample |
| `age_category` | Categorical | One-hot encode (puppy/young/adult/senior) — soft filter; replace with `age_years_approx` once derived |
| `gender` | Categorical | One-hot encode (male/female) |
| `breed_primary` | Categorical | Group into ~10 breed families; one-hot if <30 unique |
| `coat_length` | Categorical | One-hot encode (Short/Medium/Curly); null → "Unknown" category |
| `color` | Categorical | One-hot encode (11 values); null → "Unknown" category; soft filter only |
| `weight_min` | Numeric | Use directly as range filter lower bound |
| `weight_max` | Numeric | Use directly as range filter upper bound |
| `is_mixed` | Boolean | Binary (0/1); always populated |
| `special_needs` | Boolean | Binary (0/1); always populated |
| `vaccinated` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `spayed_neutered` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `good_with_kids` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `good_with_dogs` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `good_with_cats` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `house_trained` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `personality_traits` | List of strings | Multi-hot encode (one column per trait, 1 if dog has it) |
| `description` | Free text | Sentence embedding (384-dim vector) using sentence-transformers |